In [1]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets
from torch import nn
import torch.optim as optim
from torch.nn.modules.conv import Conv2d

In [2]:
import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\carol\_netrc
wandb: Currently logged in as: cdphillips04 (cdphillips04-william-mary) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

**Load Data**

In [3]:
# preprocessing for the transforms
# can do rotation, flips, normalize
# done to all the images before training
# MNIST only has 1 channel!!
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [4]:
# path to data folder
PATH = 'C:/Users/carol/Documents/data_mining/hw3/data'

# load MNIST images
train_data = datasets.MNIST(root = PATH, train = True, download = True, transform = train_transform)
test_data = datasets.MNIST(root = PATH, train = False, download = True, transform = test_transform)

In [5]:
# data loaders for training and test data
train_loader = DataLoader(train_data, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 64, shuffle = True)

In [6]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU device: NVIDIA GeForce RTX 4070 Laptop GPU


**Write CNN Model**

In [7]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size = 3, padding = 1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),   # 14x14
        nn.Dropout(0.5),

        nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 2, stride = 2),   # 7x7
        nn.Dropout(0.5),

        nn.Conv2d(64, 64, kernel_size = 3, padding = 1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size = 1, stride = 2),    # 4x4
        nn.Dropout(0.5)
    )

    self.fc = nn.Sequential(
        nn.Linear(64*4*4, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

  def forward(self, x):
    x_out = self.conv(x)
    x = torch.flatten(x_out, 1)
    result = self.fc(x)
    return result

model = CNN()
model.to(device)

CNN(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.5, inplace=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.5, inplace=False)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU()
    (13): MaxPool2d(kernel_size=1, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Dropout(p=0.5, inplace=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=1024, ou

**Define loss function**

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

**Training and testing the model**

In [9]:
# training
with wandb.init(project = 'CNN', name = 'cnn_hw3'):
    epochs = 10
    for epoch in range(epochs):
      model.train()
      for i, (images, targets) in enumerate(train_loader):
        images, targets = images.to(device), targets.to(device)
          
        # predictions
        outputs = model(images)
        # calculate loss
        loss = criterion(outputs, targets)
    
        # back propagation and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        wandb.log({'epoch': epoch, 'loss': loss})
        if (i+1)%200 == 0:
          print('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'.format(epoch + 1, epochs, i + 1, len(train_loader), loss.item()))

Epoch [1/10], Step [200/938], Loss: 0.5616
Epoch [1/10], Step [400/938], Loss: 0.5695
Epoch [1/10], Step [600/938], Loss: 0.3422
Epoch [1/10], Step [800/938], Loss: 0.3544
Epoch [2/10], Step [200/938], Loss: 0.2452
Epoch [2/10], Step [400/938], Loss: 0.3185
Epoch [2/10], Step [600/938], Loss: 0.1455
Epoch [2/10], Step [800/938], Loss: 0.2529
Epoch [3/10], Step [200/938], Loss: 0.1608
Epoch [3/10], Step [400/938], Loss: 0.2675
Epoch [3/10], Step [600/938], Loss: 0.1934
Epoch [3/10], Step [800/938], Loss: 0.2748
Epoch [4/10], Step [200/938], Loss: 0.3538
Epoch [4/10], Step [400/938], Loss: 0.2559
Epoch [4/10], Step [600/938], Loss: 0.1060
Epoch [4/10], Step [800/938], Loss: 0.1291
Epoch [5/10], Step [200/938], Loss: 0.2357
Epoch [5/10], Step [400/938], Loss: 0.3384
Epoch [5/10], Step [600/938], Loss: 0.2788
Epoch [5/10], Step [800/938], Loss: 0.2624
Epoch [6/10], Step [200/938], Loss: 0.3725
Epoch [6/10], Step [400/938], Loss: 0.2710
Epoch [6/10], Step [600/938], Loss: 0.0784
Epoch [6/10

epoch,▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
loss,█▃▂▃▃▂▂▂▁▂▁▁▂▂▁▂▂▂▂▁▂▂▁▁▁▁▁▂▁▂▁▁▁▁▂▂▁▁▁▁
epoch,9
loss,0.02056


In [10]:
# testing
model.eval()
with torch.no_grad():
  correct = 0
  total = 0
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
      
    # predictions
    outputs = model(images)

    _, predicted = outputs.max(1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print('Model accuracy for test data: {}%'.format(100*correct/total))

Model accuracy for test data: 98.3%
